# LingBot-Map on Colab — video → scene.glb (+ optional flythrough)

This is the model that matches the **LingBot** look (not VGGT).

| Output | How |
|--------|-----|
| `scene.glb` | LingBot-Map reconstruction (same as official `demo.py` → GLB) |
| `flythrough.mp4` | Optional: open the GLB in Streamlit / a viewer, or use LingBot `demo_render` later |

## Run order
1. **Runtime → Change runtime type → T4 GPU**
2. Run steps top to bottom
3. Download `scene.glb`
4. Attach it in your Streamlit app


## 0. Check GPU


In [ ]:
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU, then Runtime → Run all.")
print("gpu", torch.cuda.get_device_name(0))


## 1. Install LingBot-Map


In [ ]:
import os
from pathlib import Path

%cd /content
if not Path("/content/lingbot-map").exists():
    !git clone --depth 1 https://github.com/Robbyant/lingbot-map.git
%cd /content/lingbot-map
!pip -q install -e ".[vis]" huggingface_hub
print("cwd", os.getcwd())


## 2. Download LingBot checkpoint from Hugging Face


In [ ]:
from huggingface_hub import hf_hub_download

MODEL_PATH = hf_hub_download(
    repo_id="robbyant/lingbot-map",
    filename="lingbot-map.pt",
    local_dir="/content/weights",
)
print("checkpoint:", MODEL_PATH)


## 3. Upload YOUR video (the same one from Streamlit)


In [ ]:
from google.colab import files
from pathlib import Path

UPLOAD_DIR = Path("/content/inputs")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
print("Upload one indoor video (mp4/mov/...)")
uploaded = files.upload()
assert uploaded, "Upload a video"
VIDEO_PATH = UPLOAD_DIR / list(uploaded.keys())[0]
VIDEO_PATH.write_bytes(uploaded[list(uploaded.keys())[0]])
# if files.upload already saved in cwd, also handle that
for name, data in uploaded.items():
    dest = UPLOAD_DIR / name
    dest.write_bytes(data)
    VIDEO_PATH = dest
    print("saved", dest, f"({len(data)/1e6:.2f} MB)")


## 4. Reconstruct with LingBot-Map → `scene.glb`

Same path as the official repo: frames → GCTStream → `predictions_to_glb`.

If you run out of memory on T4, set `FIRST_K = 24`.


In [ ]:
from tqdm.std import tqdm as std_tqdm
import argparse
import sys
import time
from pathlib import Path

import torch

sys.path.insert(0, "/content/lingbot-map")
import demo as lingbot_demo
lingbot_demo.tqdm = std_tqdm  # Colab-safe progress bars
from lingbot_map.vis.glb_export import predictions_to_glb

FIRST_K = 48   # max frames after FPS sampling; lower if OOM
FPS = 10
CONF_THRES = 50.0
OUT_GLB = Path("/content/scene.glb")

device = torch.device("cuda")
print("video:", VIDEO_PATH)

images, _, _ = lingbot_demo.load_images(
    video_path=str(VIDEO_PATH),
    fps=FPS,
    first_k=FIRST_K,
    image_size=518,
    patch_size=14,
)
num_frames = int(images.shape[0])
scale_frames = min(8, max(1, num_frames - 1))
print(f"frames={num_frames} scale_frames={scale_frames}")

args = argparse.Namespace(
    mode="streaming",
    model_path=str(MODEL_PATH),
    image_size=518,
    patch_size=14,
    enable_3d_rope=True,
    max_frame_num=1024,
    kv_cache_sliding_window=64,
    num_scale_frames=scale_frames,
    use_sdpa=True,              # no FlashInfer needed on Colab
    camera_num_iterations=4,
    window_size=64,
    overlap_size=16,
    overlap_keyframes=None,
)
model = lingbot_demo.load_model(args, device)
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
if getattr(model, "aggregator", None) is not None:
    model.aggregator = model.aggregator.to(dtype=dtype)

images = images.to(device)
t0 = time.time()
with torch.no_grad(), torch.amp.autocast("cuda", dtype=dtype):
    predictions = model.inference_streaming(
        images,
        num_scale_frames=scale_frames,
        keyframe_interval=1,
        output_device=torch.device("cpu"),
    )
print(f"inference {time.time()-t0:.1f}s")

images_for_post = predictions.get("images", images)
predictions, images_cpu = lingbot_demo.postprocess(predictions, images_for_post)
vis = lingbot_demo.prepare_for_visualization(predictions, images_cpu)

# show_cam=False avoids a wireframe-only look in some viewers
scene = predictions_to_glb(vis, conf_thres=CONF_THRES, show_cam=False, mask_sky=False)
scene.export(str(OUT_GLB))
print(f"wrote {OUT_GLB} ({OUT_GLB.stat().st_size/1e6:.2f} MB)")


## 5. Download the LingBot GLB

Open it in https://gltf-viewer.donmccurdy.com/  
Then attach `scene.glb` in Streamlit.

### Flythrough note
LingBot’s official **moving** flythrough comes from `demo_render/batch_demo.py` (heavier: Kaolin + CUDA extensions).
For the internship UI now: use this high-quality **GLB** as the main LingBot output.
A proper LingBot flythrough MP4 can be added as a later Colab step once GLB works.


In [ ]:
from google.colab import files
from pathlib import Path
assert Path("/content/scene.glb").is_file()
files.download("/content/scene.glb")
print("Attach scene.glb in Streamlit for your job.")
